<a href="https://colab.research.google.com/github/tanviengineer/ai-learning-journey/blob/main/NLP10_Sentimental_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Import required libraries
import numpy as np
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# spaCy
import spacy

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# 2. DOWNLOAD NLTK RESOURCES
# --------------------------------------------------

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
# 3. LOAD SPACY MODEL
# --------------------------------------------------

# Install first if required:
# !pip install spacy
# !python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

In [ ]:
# 4. LOAD DATASET
# --------------------------------------------------

# If using Google Colab, upload the extracted CSV file
# and provide its path here.

file_path = "twitter_training.csv"

df = pd.read_csv(file_path, header=None)

In [ ]:
df.columns = ["ID", "Topic", "Sentiment", "Text"]

# 6. BASIC DATASET INFORMATION
# --------------------------------------------------

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSentiment Distribution:")
print(df["Sentiment"].value_counts())


Dataset Shape:
(74682, 4)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         74682 non-null  int64 
 1   Topic      74682 non-null  object
 2   Sentiment  74682 non-null  object
 3   Text       73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB
None

Missing Values:
ID             0
Topic          0
Sentiment      0
Text         686
dtype: int64

Sentiment Distribution:
Sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [ ]:
df.sample(10)

,ID,Topic,Sentiment,Text,Clean_Text
11185,13123,Xbox(Xseries),Positive,After 15 tense minutes of waiting for my,tense minute waiting
28583,518,ApexLegends,Negative,"literally poor bro, left this game when i was ...","literally poor bro, left game clear re him. kn..."
67719,7198,johnson&johnson,Negative,MAYBE JOHNSON... JOHNSON SHOULD PUT HIS MONEY ...,maybe johnson... johnson put money cancer caus...
1154,2603,Borderlands,Positive,"So, after the last 9 days of streaming on the ...","so, last day streaming rebound, last night stu..."
41939,1595,Battlefield,Positive,have true love!,true love!
50212,6220,FIFA,Neutral,"@EAHelp @EAFIFAMOBILE Hi, actually I wanted to...","eahelp eafifamobile hi, actually wanted help s..."
6315,286,Amazon,Neutral,amazon.in/Shivaji-Great-…. Maharashtra No grea...,amazon.in/shivaji-great-…. maharashtra great. ...
18823,12426,WorldOfCraft,Neutral,I just earned the [Horrific Vision of Orgrimma...,earned [horrific vision orgrimmar] achievement!
68390,3711,Cyberpunk2077,Negative,Fuck u,fuck
52741,10660,RedDeadRedemption(RDR),Neutral,Red Dead Redemption 2 PC gameplay. Robbery of ...,red dead redemption pc gameplay. robbery train...


In [ ]:
# 7. REMOVE MISSING VALUES
# --------------------------------------------------

df = df.dropna(subset=["Text", "Sentiment"])

print("\nDataset shape after removing missing values:")
print(df.shape)



Dataset shape after removing missing values:
(73996, 4)


In [ ]:
# 8. REMOVE IRRELEVANT SENTIMENTS
# --------------------------------------------------

df = df[df["Sentiment"].isin([
    "Positive",
    "Negative",
    "Neutral"
])]

print("\nSentiment distribution after removing Irrelevant:")
print(df["Sentiment"].value_counts())


Sentiment distribution after removing Irrelevant:
Sentiment
Negative    22358
Positive    20655
Neutral     18108
Name: count, dtype: int64


In [ ]:
# 9. TEXT PREPROCESSING FUNCTION
# --------------------------------------------------
stop_words = str(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # convert text to string
    text = str(text)

    # convert to lower case
    text = text.lower()

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@", "", text)

    # remove hashtag symbol but keep the word
    text = re.sub(r"#", "", text)

    # remove numbers
    text = re.sub(r"\d+", "", text)
    # remove numbers
    text = re.sub(r"\d+", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # tokenization
    words = text.split()

    # remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Lemmatizing using NLTK
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


In [ ]:

# APPLY NLTK TEXT PREPROCESSING
df["Clean_Text"] = df["Text"].apply(clean_text)

/tmp/ipykernel_2298/3167502636.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Clean_Text"] = df["Text"].apply(clean_text)


In [ ]:
# DISPLAY ORIGINAL AND CLEANED TEXT
print("\nOriginal and cleaned Text:")
print(df[["Text", "Clean_Text"]].head(10))


Original and cleaned Text:
                                                Text  \
0  im getting on borderlands and i will murder yo...   
1  I am coming to the borders and I will kill you...   
2  im getting on borderlands and i will kill you ...   
3  im coming on borderlands and i will murder you...   
4  im getting on borderlands 2 and i will murder ...   
5  im getting into borderlands and i can murder y...   
6  So I spent a few hours making something for fu...   
7  So I spent a couple of hours doing something f...   
8  So I spent a few hours doing something for fun...   
9  So I spent a few hours making something for fu...   

                                          Clean_Text  
0                          getting borderland murder  
1                            coming border kill all,  
2                       getting borderland kill all,  
3                      coming borderland murder all,  
4                     getting borderland murder all,  
5                     get

In [ ]:
#11.  REMOVE EMPTY TEXT RECORDS
df = df[df["Clean_Text"].str.strip() != ""]

In [ ]:
# 12 optional spacy preprocessing
def spacy_preprocess(text):
  doc =nlp(text)
  tokens =[]
  for token in doc :
    if (
        not token.is_stop
        and not token.is_punct
        and not token.is_space
    ):
        tokens.append(token.lemma_.lower())
    return "".join(tokens)
  #  apply(spacy_preprocess
df["Processed_Text"]= df["Clean_Text"].apply(spacy_preprocess)
# display Processed_Text
print("\n Text after spacy preprocessing : ")
print(df[["Text","Processed_Text"]].head(10))




 Text after spacy preprocessing : 
                                                Text Processed_Text
0  im getting on borderlands and i will murder yo...            get
1  I am coming to the borders and I will kill you...           come
2  im getting on borderlands and i will kill you ...        getting
3  im coming on borderlands and i will murder you...           come
4  im getting on borderlands 2 and i will murder ...            get
5  im getting into borderlands and i can murder y...            get
6  So I spent a few hours making something for fu...          spend
7  So I spent a couple of hours doing something f...          spend
8  So I spent a few hours doing something for fun...          spend
9  So I spent a few hours making something for fu...          spend


In [ ]:
# 13 define featureas and target
x= df["Processed_Text"]
y =df["Sentiment"]

In [ ]:
X_train ,X_test ,Y_train ,Y_test = train_test_split(x,y,test_size =0.20,random_state=42,stratify=y)
print("\n Training samples:" ,len(X_train))
print("Testing samples: ",len(X_test))


 Training samples: 47914
Testing samples:  11979


In [ ]:
vectorizer =TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf=vectorizer.fit_transform(X_train)

X_test_tfidf =vectorizer.transform(X_test)
print("\n TF-IDF Training Shape : ")
print(X_train_tfidf.shape)
print("\n TF-IDF Testing Shape : ")
print(X_test_tfidf.shape)


 TF-IDF Training Shape : 
(47914, 2675)

 TF-IDF Testing Shape : 
(11979, 2675)
